In [ ]:
import asyncio
import os
import json
import time
import smtplib
import schedule
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta_classic as ta
import gspread
import telegram
import requests
from bs4 import BeautifulSoup
from email.message import EmailMessage
from oauth2client.service_account import ServiceAccountCredentials
from dotenv import load_dotenv

# --- 0. INITIAL SETUP & SECURITY ---
load_dotenv()

class CryptoBot:
    def __init__(self):
        # Sensitive Credentials
        self.TELEGRAM_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN')
        self.CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')
        self.GMAIL_USER = os.getenv('GMAIL_ADDRESS')
        self.GMAIL_PASS = os.getenv('GMAIL_APP_PASSWORD')
        self.SERVICE_ACCOUNT = os.getenv('GCP_SERVICE_ACCOUNT_FILE')
        self.CMC_API_KEY = os.getenv('CMC_API_KEY') # NEW: CoinMarketCap Key
        
        # Config & Portfolio
        self.CONFIG_CACHE_FILE = "config_cache.json"
        self.config = self.load_config()
        self.cash_usd = float(self.config.get('BUDGET_USD', 10000))
        self.btc_balance = 0.0
        self.last_dca_price = None
        self.active_trades = []
        self.trade_history = []
        
        if self.TELEGRAM_TOKEN:
            self.bot = telegram.Bot(token=self.TELEGRAM_TOKEN)
        print("🚀 Smart ₿itcoin Trading System Initialized.")

    # --- 1. CONFIGURATION MANAGEMENT ---
    def load_config(self):
        try:
            if not self.SERVICE_ACCOUNT: raise ValueError("No GCP file in .env")
            scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
            creds = ServiceAccountCredentials.from_json_keyfile_name(self.SERVICE_ACCOUNT, scope)
            client = gspread.authorize(creds)
            sheet = client.open("CryptoBot_Config").sheet1
            data = sheet.get_all_records()
            new_config = {row['Key']: row['Value'] for row in data}
            with open(self.CONFIG_CACHE_FILE, 'w') as f: json.dump(new_config, f)
            return new_config
        except Exception as e:
            print(f"⚠️ Config Warning: {e}. Using local cache.")
            with open(self.CONFIG_CACHE_FILE, 'r') as f: return json.load(f)

    # --- 2. MULTI-SOURCE DATA FETCHING ---
    
    def fetch_investing_sentiment(self):
        """Scrapes Investing.com for basic sentiment (Mock Implementation)"""
        try:
            # Note: Investing.com has heavy anti-scraping. In production, use headers.
            url = "https://www.investing.com/crypto/bitcoin/historical-data"
            headers = {'User-Agent': 'Mozilla/5.0'}
            # response = requests.get(url, headers=headers)
            # soup = BeautifulSoup(response.content, 'html.parser')
            # Scrape logic here...
            return "Neutral/Bullish" 
        except: return "Unknown"

    def get_cmc_data(self):
        """Fetches Global Market Cap from CoinMarketCap API"""
        if not self.CMC_API_KEY: return {"market_cap_dominance": 50.0}
        url = "https://pro-api.coinmarketcap.com/v1/global-metrics/quotes/latest"
        headers = {'X-CMC_PRO_API_KEY': self.CMC_API_KEY}
        try:
            response = requests.get(url, headers=headers).json()
            return response['data']['btc_dominance']
        except: return 50.0

    def fetch_and_engineer(self):
        """Combines yfinance OHLCV with Multi-Source Metadata"""
        print("📊 Fetching market data from Yahoo Finance...")
        df = yf.download("BTC-USD", period="7d", interval="1h", progress=False)
        
        # FIX: Flatten MultiIndex columns (Fixes the AttributeError)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Technical Indicators
        df.ta.atr(length=14, append=True)
        df.ta.rsi(length=14, append=True)
        
        # Add External Data as Features
        df['btc_dominance'] = self.get_cmc_data()
        df['sentiment'] = self.fetch_investing_sentiment()
        
        return df

    # --- 3. EXECUTION ENGINE (Targeting Coinbase/Binance) ---
    def execute_logic(self):
        data = self.fetch_and_engineer()
        current_price = float(data['Close'].iloc[-1])
        current_atr = float(data['ATRr_14'].iloc[-1])
        
        self.check_stop_losses(current_price, current_atr)
        
        # DCA Logic
        dca_trigger = float(self.config.get('DCA_DROP_PERCENT', 0.03))
        if self.last_dca_price is None or (self.last_dca_price - current_price) / self.last_dca_price >= dca_trigger:
            self.buy_btc(float(self.config.get('DCA_AMOUNT', 500)), current_price, "DCA Buy Triggered")
            self.last_dca_price = current_price

    def buy_btc(self, usd_amount, price, reason):
        """In production, this calls Coinbase Advanced Trade API"""
        if self.cash_usd >= usd_amount:
            # API CALL WOULD GO HERE: coinbase.place_order(amount=usd_amount, side='buy')
            btc_bought = usd_amount / price
            self.cash_usd -= usd_amount
            self.btc_balance += btc_bought
            msg = f"💰 {reason}: Bought {btc_bought:.6f} BTC at ${price:,.2f}"
            self.active_trades.append({'entry_price': price, 'amount': btc_bought})
            self.notify_trade(msg)

    def check_stop_losses(self, current_price, atr):
        k = float(self.config.get('ATR_K_MULTIPLIER', 1.5))
        for trade in self.active_trades[:]:
            stop_price = trade['entry_price'] - (k * atr)
            if current_price <= stop_price:
                # API CALL WOULD GO HERE: coinbase.place_order(amount=trade['amount'], side='sell')
                self.btc_balance -= trade['amount']
                self.cash_usd += (trade['amount'] * current_price)
                msg = f"🛑 Stop-Loss Hit: Sold at ${current_price:,.2f}"
                self.notify_trade(msg)
                self.active_trades.remove(trade)

    # --- 4. MONITORING ---
    def notify_trade(self, message):
        print(message)
        if self.TELEGRAM_TOKEN and self.CHAT_ID:
            try:
                # This creates a small loop just to send this one message
                asyncio.create_task(self.bot.send_message(chat_id=self.CHAT_ID, text=message))
            except Exception as e:
                print(f"Telegram fail: {e}")

    def send_weekly_email(self):
        try:
            msg = EmailMessage()
            content = f"Weekly Report\nPortfolio: ${self.cash_usd + (self.btc_balance * 60000):,.2f}"
            msg.set_content(content)
            msg['Subject'] = "MonReader: Weekly Trading Summary"
            msg['From'] = self.GMAIL_USER
            msg['To'] = self.GMAIL_USER
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
                smtp.login(self.GMAIL_USER, self.GMAIL_PASS)
                smtp.send_message(msg)
            print("📧 Weekly Report Sent.")
        except Exception as e: print(f"Email failed: {e}")

# --- 5. MAIN LOOP ---
if __name__ == "__main__":
    agent = CryptoBot()
    schedule.every().monday.at("09:00").do(agent.send_weekly_email)
    
    while True:
        try:
            agent.config = agent.load_config()
            agent.execute_logic()
            schedule.run_pending()
            print("Loop complete. Sleeping 30m...")
            time.sleep(1800)
        except Exception as e:
            print(f"Main Loop Error: {e}")
            time.sleep(60)

⚠️ Config Warning: No GCP file in .env. Using local cache.
🚀 Smart ₿itcoin Trading System Initialized.
⚠️ Config Warning: No GCP file in .env. Using local cache.
📊 Fetching market data from Yahoo Finance...
💰 DCA Buy Triggered: Bought 0.007558 BTC at $66,152.13
Loop complete. Sleeping 30m...
